In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[0]))

import scripts.prepare as prepare
import scripts.services as services
import scripts.roi as roi_services
import scripts.bss_pipeline as bss_pl
import scripts.visualization as vis

%load_ext autoreload
%autoreload 2

### fixed variables

In [ ]:
N_PC = 10 # discovered by analysis
NFRAMES_OUT = 1200
SCALE_OUT = 0.2
FPS_OUT = 240
N_PEAKS = 5

VIDEO_PATH_IN = str(Path.cwd().parents[0] / "videos/camp_3/20260527/20260527_2/m/VID_20260527_074035158.mp4")
VIDEO_NAME = VIDEO_PATH_IN.split("/")[-1]
VIDEO_PATH_OUT = str(Path.cwd().parents[0] / "outputs/misc" / VIDEO_NAME)
SOURCES_PATH = str(Path.cwd().parents[0] / "outputs/sources")

ROIS = roi_services.load_rois(str(Path.cwd().parents[0] / "rois/rois.json"))
VIDEO_ROI = ROIS[VIDEO_NAME]

video_in_info = services.get_video_info(VIDEO_PATH_IN)
video_in_info

### pre-processing

In [ ]:
prepare.pre_processing(VIDEO_PATH_IN, VIDEO_PATH_OUT, NFRAMES_OUT, FPS_OUT, SCALE_OUT, VIDEO_ROI)

In [ ]:
video_out_info = services.get_video_info(VIDEO_PATH_OUT)
video_out_info

### computer vision

In [ ]:
unmixed, _, W, H = bss_pl.run_pipeline(VIDEO_PATH_OUT, N_PC)
mixtures = H[:, :N_PC]

In [ ]:
spatial_maps, A = services.compute_source_spatial_maps(W, mixtures, unmixed, N_PC)

#### Create static source overlay video

In [ ]:
services.create_static_source_overlay_video(VIDEO_PATH_OUT, SOURCES_PATH + "/static_source_03_overlay.mp4", spatial_maps[:, 8])

In [ ]:
for source_idx in range(N_PC):
    services.create_static_source_overlay_video(VIDEO_PATH_OUT, SOURCES_PATH + f"/static_source_{source_idx:02d}_overlay.mp4", spatial_maps[:, source_idx])

#### Create dynamic source overlay video

In [ ]:
services.create_dynamic_source_overlay_video(VIDEO_PATH_OUT, SOURCES_PATH + "/dynamic_source_xx_overlay.mp4", spatial_maps[:, 9], unmixed[:, 9])

In [ ]:
for source_idx in range(N_PC):
    services.create_dynamic_source_overlay_video(VIDEO_PATH_OUT, SOURCES_PATH + f"/dynamic_source_{source_idx:02d}_overlay.mp4", spatial_maps[:, source_idx], unmixed[:, source_idx])

#### Create video grid

In [ ]:
video_paths = [SOURCES_PATH + f"/static_source_{i:02d}_overlay.mp4" for i in [1, 2, 3]]
#video_paths = [SOURCES_PATH + f"/static_source_{i:02d}_overlay.mp4" for i in range(N_PC)]
#video_paths = [SOURCES_PATH + f"/dynamic_source_{i:02d}_overlay.mp4" for i in range(N_PC)]

In [ ]:
services.create_video_grid(video_paths, SOURCES_PATH + "/sources_grid.mp4", n_cols=3, cell_width=320, titles = ["09", "09", "08"])